# Bluesky Alt Text Dataset

279,196 image alt-text pairs from 489 Bluesky accounts, validated for quality.

This notebook walks through the dataset: loading, basic stats, distributions, and a few things you can do with it.

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#0d1117',
    'axes.edgecolor': '#30363d',
    'axes.labelcolor': '#c9d1d9',
    'text.color': '#c9d1d9',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'grid.color': '#21262d',
    'font.size': 11,
})

## Load the data

In [ ]:
# Works with both .jsonl and .jsonl.gz
df = pd.read_json("corpus.jsonl.gz", lines=True)
print(f"{len(df):,} rows, {df['author_handle'].nunique()} authors")
df.head(3)

In [ ]:
df.dtypes

## Alt text length distribution

The core question: how long are these descriptions?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax = axes[0]
df['image_alt_length'].clip(upper=1000).hist(
    bins=80, ax=ax, color='#58a6ff', edgecolor='#0d1117', alpha=0.9
)
ax.axvline(df['image_alt_length'].median(), color='#f0883e', linestyle='--', label=f"Median: {df['image_alt_length'].median():.0f}")
ax.axvline(df['image_alt_length'].mean(), color='#3fb950', linestyle='--', label=f"Mean: {df['image_alt_length'].mean():.0f}")
ax.set_xlabel('Alt text length (characters)')
ax.set_ylabel('Count')
ax.set_title('Alt text length distribution (clipped at 1000)')
ax.legend()
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

# Box plot of length by bucket
ax = axes[1]
buckets = pd.cut(df['image_alt_length'], bins=[0, 50, 100, 200, 500, 1000, 60000],
                 labels=['<50', '50-100', '100-200', '200-500', '500-1K', '1K+'])
bucket_counts = buckets.value_counts().sort_index()
bars = ax.bar(bucket_counts.index.astype(str), bucket_counts.values, color='#58a6ff', edgecolor='#0d1117')
ax.set_xlabel('Alt text length range')
ax.set_ylabel('Count')
ax.set_title('Rows by length bucket')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
for bar, val in zip(bars, bucket_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'{val/len(df)*100:.0f}%', ha='center', va='bottom', fontsize=9, color='#8b949e')

plt.tight_layout()
plt.show()

## Top contributors

In [ ]:
top = df.groupby('author_handle').agg(
    rows=('alt_text', 'count'),
    mean_length=('image_alt_length', 'mean'),
    median_length=('image_alt_length', 'median'),
).sort_values('rows', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
top20 = top.head(20)
bars = ax.barh(top20.index[::-1], top20['rows'][::-1], color='#58a6ff', edgecolor='#0d1117')
ax.set_xlabel('Number of alt-text rows')
ax.set_title('Top 20 contributors')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1000:.1f}K'))
plt.tight_layout()
plt.show()

print(f"\n{len(top)} total authors")
top.head(20).style.format({'mean_length': '{:.0f}', 'median_length': '{:.0f}'})

## Language breakdown

In [ ]:
# Explode the langs_json field
langs = df['langs_json'].apply(json.loads).explode().dropna()
lang_counts = langs.value_counts().head(10)

fig, ax = plt.subplots(figsize=(8, 4))
lang_counts.plot.barh(ax=ax, color='#58a6ff', edgecolor='#0d1117')
ax.set_xlabel('Rows')
ax.set_title('Top 10 languages')
ax.invert_yaxis()
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.show()

## Alt text quality per author

How consistent is the quality across accounts?

In [ ]:
author_stats = df.groupby('author_handle').agg(
    n=('alt_text', 'count'),
    mean_len=('image_alt_length', 'mean'),
    median_len=('image_alt_length', 'median'),
    std_len=('image_alt_length', 'std'),
)

fig, ax = plt.subplots(figsize=(10, 6))
sc = ax.scatter(
    author_stats['n'],
    author_stats['mean_len'],
    s=author_stats['n'].clip(upper=2000) / 20,
    c=author_stats['median_len'],
    cmap='coolwarm',
    alpha=0.7,
    edgecolors='#30363d',
    linewidth=0.5,
)
ax.set_xlabel('Number of rows')
ax.set_ylabel('Mean alt text length (chars)')
ax.set_title('Author quality: volume vs. description length')
ax.set_xscale('log')
plt.colorbar(sc, label='Median alt text length')
plt.tight_layout()
plt.show()

## Temporal coverage

When were these posts created?

In [ ]:
df['created_date'] = pd.to_datetime(df['created_at'], errors='coerce')
monthly = df.set_index('created_date').resample('M').size()

fig, ax = plt.subplots(figsize=(12, 4))
monthly.plot(ax=ax, color='#58a6ff', linewidth=1.5)
ax.fill_between(monthly.index, monthly.values, alpha=0.15, color='#58a6ff')
ax.set_ylabel('Posts with alt text')
ax.set_title('Posts over time')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1000:.1f}K'))
plt.tight_layout()
plt.show()

## Sample alt text

What does the actual text look like?

In [ ]:
# Sample 10 rows with long, descriptive alt text
long_alt = df[df['image_alt_length'].between(150, 500)].sample(10, random_state=42)
for _, row in long_alt.iterrows():
    print(f"@{row['author_handle']} ({row['image_alt_length']} chars):")
    print(f"  {row['alt_text'][:300]}")
    print()

## Images per post

Bluesky allows up to 4 images per post. How are they distributed?

In [ ]:
img_counts = df['image_count_in_post'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(6, 4))
img_counts.plot.bar(ax=ax, color='#58a6ff', edgecolor='#0d1117')
ax.set_xlabel('Images per post')
ax.set_ylabel('Rows')
ax.set_title('Image count distribution')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))
for i, (idx, val) in enumerate(img_counts.items()):
    ax.text(i, val + 500, f'{val/len(df)*100:.0f}%', ha='center', fontsize=9, color='#8b949e')
plt.tight_layout()
plt.show()